# Week 6: Movement Assessment and Motion Capture
## Digital Healthcare & Physical Therapy

**Learning Objectives:**
- Joint Angle Analysis
- Range of Motion (ROM) Measurement
- Normal vs. Post-Treatment Movement Comparison
- Clinical Movement Quality Assessment

---
> **Colab Instructions:** Run cells top to bottom. Google Drive mount is optional — outputs also save locally to `/content/outputs/`.

In [ ]:
# ============================================================
# CELL 1: Setup — Install & Import Libraries
# ============================================================
# All required libraries are pre-installed in Colab.
# Run this cell first.

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import seaborn as sns
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ---- Font fix: use a Unicode-safe font ----
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# ---- Output directory (local, no Drive needed) ----
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = f'/content/outputs/run_{timestamp}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Libraries loaded successfully.')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELL 2 (Optional): Mount Google Drive
# ============================================================
# Uncomment and run this cell if you want to save outputs to Google Drive.

# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_BASE = '/content/drive/MyDrive/MotionAssessment'
# os.makedirs(DRIVE_BASE, exist_ok=True)
# OUTPUT_DIR = f'{DRIVE_BASE}/run_{timestamp}'
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# print(f'Drive output directory: {OUTPUT_DIR}')

---
## Part 1 — Beginner: Single Joint Angle Analysis (Knee Flexion)

In [ ]:
# ============================================================
# CELL 3: Generate Knee Angle Data
# ============================================================
np.random.seed(42)

# Parameters
SAMPLING_RATE = 100   # Hz
GAIT_DURATION = 1.0   # 1 second per gait cycle (120 steps/min)
NUM_CYCLES    = 10    # Number of gait cycles to simulate

t           = np.arange(0, NUM_CYCLES, 1 / SAMPLING_RATE)
gait_phase  = (t % GAIT_DURATION) / GAIT_DURATION          # 0 → 1 per cycle

# Knee angle biomechanical model
# 0 %  : Initial Contact   ~5°
# 50%  : Mid-Stance        ~20°
# 100% : Toe-Off           ~5°
knee_angle = 5 + 15 * np.sin(np.pi * gait_phase) + 1.0 * np.random.randn(len(t))

knee_df = pd.DataFrame({
    'time':       t,
    'gait_phase': gait_phase * 100,   # Convert to 0–100 %
    'knee_angle': knee_angle
})

print('Knee angle data generated.')
print(knee_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# CELL 4: ROM Calculation
# ============================================================
NORMAL_KNEE_ROM = 60  # degrees (clinical reference)

knee_max = knee_df['knee_angle'].max()
knee_min = knee_df['knee_angle'].min()
knee_rom = knee_max - knee_min

rom_df = pd.DataFrame({
    'Measure':  ['Maximum Angle', 'Minimum Angle', 'Measured ROM', 'Normal ROM', 'ROM (% of normal)'],
    'Value':    [knee_max, knee_min, knee_rom, NORMAL_KNEE_ROM, (knee_rom / NORMAL_KNEE_ROM) * 100],
    'Unit':     ['deg', 'deg', 'deg', 'deg', '%']
})

print('\n=== Knee ROM Analysis ===')
print(rom_df.to_string(index=False, float_format='%.2f'))

In [ ]:
# ============================================================
# CELL 5: Visualise Knee Joint Angle
# ============================================================
one_cycle = knee_df[knee_df['time'] < GAIT_DURATION].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full time-series
axes[0].plot(knee_df['time'], knee_df['knee_angle'], color='steelblue', lw=1.2, alpha=0.85)
axes[0].axhline(knee_max, color='red',    ls='--', lw=1, label=f'Max {knee_max:.1f}deg')
axes[0].axhline(knee_min, color='orange', ls='--', lw=1, label=f'Min {knee_min:.1f}deg')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Knee Angle (deg)')
axes[0].set_title('Knee Angle — Full Recording')
axes[0].legend()

# Right: one normalised gait cycle
axes[1].fill_between(one_cycle['gait_phase'], one_cycle['knee_angle'],
                     alpha=0.25, color='steelblue')
axes[1].plot(one_cycle['gait_phase'], one_cycle['knee_angle'],
             color='steelblue', lw=2.5, marker='o', markersize=3)
axes[1].set_xlabel('Gait Cycle (%)')
axes[1].set_ylabel('Knee Angle (deg)')
axes[1].set_title('Knee Angle — Single Gait Cycle')

# Annotate gait phases
for pct, label in [(0, 'Initial\nContact'), (50, 'Mid\nStance'), (100, 'Toe\nOff')]:
    axes[1].axvline(pct, color='gray', ls=':', lw=1)
    axes[1].text(pct + 1, axes[1].get_ylim()[1] * 0.95, label,
                fontsize=8, color='gray', va='top')

plt.tight_layout()
save_path = f'{OUTPUT_DIR}/01_knee_angle_analysis.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Figure saved: {save_path}')
plt.show()

# Save CSV
knee_df.to_csv(f'{OUTPUT_DIR}/01_knee_angle_data.csv', index=False)
rom_df.to_csv(f'{OUTPUT_DIR}/01_rom_analysis.csv', index=False)
print('CSV files saved.')

---
## Part 2 — Intermediate: Multi-Joint Kinematics (Hip, Knee, Ankle)

In [ ]:
# ============================================================
# CELL 6: Generate Multi-Joint Data
# ============================================================
np.random.seed(0)
n = len(t)

hip_angle    = 20 * np.sin(2 * np.pi * gait_phase - np.pi/4) + 1.5 * np.random.randn(n)
knee_angle2  =  5 + 15 * np.sin(np.pi * gait_phase)          + 1.0 * np.random.randn(n)
ankle_angle  = 10 * np.sin(2 * np.pi * gait_phase + np.pi/3) + 0.8 * np.random.randn(n)

multi_df = pd.DataFrame({
    'time':        t,
    'gait_phase':  gait_phase * 100,
    'hip_angle':   hip_angle,
    'knee_angle':  knee_angle2,
    'ankle_angle': ankle_angle
})

# ROM per joint
joints = {'Hip': 'hip_angle', 'Knee': 'knee_angle', 'Ankle': 'ankle_angle'}
normal_roms = {'Hip': 50, 'Knee': 60, 'Ankle': 30}

rom_rows = []
for name, col in joints.items():
    jmax  = multi_df[col].max()
    jmin  = multi_df[col].min()
    jrom  = jmax - jmin
    nrom  = normal_roms[name]
    rom_rows.append({'Joint': name, 'Max': jmax, 'Min': jmin,
                     'ROM': jrom, 'Normal ROM': nrom,
                     'ROM %': (jrom / nrom) * 100})

rom_comparison = pd.DataFrame(rom_rows)
print('=== Multi-Joint ROM Comparison ===')
print(rom_comparison.to_string(index=False, float_format='%.2f'))

In [ ]:
# ============================================================
# CELL 7: Multi-Joint Kinematics Plot
# ============================================================
one = multi_df[multi_df['time'] < GAIT_DURATION].copy()

colors = {'hip_angle': '#e55', 'knee_angle': '#58a', 'ankle_angle': '#2a7'}
labels = {'hip_angle': 'Hip', 'knee_angle': 'Knee', 'ankle_angle': 'Ankle'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Individual joint plots
for ax, (col, color) in zip(axes.flat[:3], colors.items()):
    ax.fill_between(one['gait_phase'], one[col], alpha=0.2, color=color)
    ax.plot(one['gait_phase'], one[col], color=color, lw=2.5)
    ax.set_title(f'{labels[col]} Joint Angle')
    ax.set_xlabel('Gait Cycle (%)')
    ax.set_ylabel('Angle (deg)')

# Overlay
ax4 = axes[1, 1]
for col, color in colors.items():
    ax4.plot(one['gait_phase'], one[col], color=color, lw=2, label=labels[col])
ax4.set_title('All Joints — Overlay')
ax4.set_xlabel('Gait Cycle (%)')
ax4.set_ylabel('Angle (deg)')
ax4.legend()

plt.suptitle('Multi-Joint Kinematics During Gait', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save_path = f'{OUTPUT_DIR}/02_multi_joint_kinematics.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Figure saved: {save_path}')
plt.show()

multi_df.to_csv(f'{OUTPUT_DIR}/02_multi_joint_angles.csv', index=False)
rom_comparison.to_csv(f'{OUTPUT_DIR}/02_rom_comparison.csv', index=False)
print('CSV files saved.')

---
## Part 3 — Advanced: Pre vs. Post Treatment Comparison

In [ ]:
# ============================================================
# CELL 8: Generate Pre / Post Treatment Data
# ============================================================
np.random.seed(7)

def make_treatment_data(seed, hip_scale=0.8, knee_scale=0.7, ankle_scale=0.75, noise=1.5):
    """Simulate joint angle data with configurable movement impairment."""
    np.random.seed(seed)
    n = len(t)
    return pd.DataFrame({
        'time':        t,
        'gait_phase':  gait_phase * 100,
        'hip_angle':   20 * hip_scale   * np.sin(2*np.pi*gait_phase - np.pi/4) + noise * np.random.randn(n),
        'knee_angle':  (5 + 15 * knee_scale * np.sin(np.pi * gait_phase))      + noise * np.random.randn(n),
        'ankle_angle': 10 * ankle_scale * np.sin(2*np.pi*gait_phase + np.pi/3) + noise * np.random.randn(n),
    })

pre  = make_treatment_data(seed=10, hip_scale=0.55, knee_scale=0.50, ankle_scale=0.50, noise=2.5)
post = make_treatment_data(seed=20, hip_scale=0.90, knee_scale=0.88, ankle_scale=0.85, noise=1.0)

print('Pre / post treatment data generated.')
print(f"Pre  — knee ROM: {pre['knee_angle'].max()-pre['knee_angle'].min():.1f} deg")
print(f"Post — knee ROM: {post['knee_angle'].max()-post['knee_angle'].min():.1f} deg")

In [ ]:
# ============================================================
# CELL 9: Movement Quality Scoring
# ============================================================
def movement_quality_score(df, normal_roms):
    """Compute a 0-100 clinical movement quality score."""
    scores = {}
    joint_map = {'Hip': 'hip_angle', 'Knee': 'knee_angle', 'Ankle': 'ankle_angle'}

    rom_sub_scores = []
    smoothness_sub_scores = []

    for jname, col in joint_map.items():
        jrom  = df[col].max() - df[col].min()
        nrom  = normal_roms[jname]
        rom_sub_scores.append(min(100, (jrom / nrom) * 100))

        diff  = np.diff(df[col].values)
        jerk  = np.std(diff)
        smooth_score = max(0, 100 - jerk * 15)
        smoothness_sub_scores.append(smooth_score)

    scores['rom_score']        = np.mean(rom_sub_scores)
    scores['smoothness_score'] = np.mean(smoothness_sub_scores)
    scores['overall_score']    = 0.6 * scores['rom_score'] + 0.4 * scores['smoothness_score']
    return scores

pre_scores  = movement_quality_score(pre,  normal_roms)
post_scores = movement_quality_score(post, normal_roms)
improvement = post_scores['overall_score'] - pre_scores['overall_score']

print('\n=== Movement Quality Scores ===')
quality_df = pd.DataFrame({
    'Metric':      ['ROM Score', 'Smoothness Score', 'Overall Score'],
    'Pre-Tx':      [pre_scores['rom_score'],  pre_scores['smoothness_score'],  pre_scores['overall_score']],
    'Post-Tx':     [post_scores['rom_score'], post_scores['smoothness_score'], post_scores['overall_score']],
    'Improvement': [post_scores['rom_score']  - pre_scores['rom_score'],
                    post_scores['smoothness_score'] - pre_scores['smoothness_score'],
                    improvement]
})
print(quality_df.to_string(index=False, float_format='%.2f'))

In [ ]:
# ============================================================
# CELL 10: Statistical Comparison (Paired t-test)
# ============================================================
stat_rows = []
for jname, col in {'Hip': 'hip_angle', 'Knee': 'knee_angle', 'Ankle': 'ankle_angle'}.items():
    tstat, pval = stats.ttest_rel(pre[col].values, post[col].values)
    effect_d = (post[col].mean() - pre[col].mean()) / (
        np.sqrt((pre[col].std()**2 + post[col].std()**2) / 2))
    stat_rows.append({
        'Joint':    jname,
        'Pre Mean': pre[col].mean(),
        'Post Mean': post[col].mean(),
        'Δ Mean':   post[col].mean() - pre[col].mean(),
        't-stat':   tstat,
        'p-value':  pval,
        "Cohen's d": effect_d,
        'Significant': 'Yes' if pval < 0.05 else 'No'
    })

stat_df = pd.DataFrame(stat_rows)
print('\n=== Paired t-test: Pre vs. Post Treatment ===')
print(stat_df.to_string(index=False, float_format='%.3f'))

In [ ]:
# ============================================================
# CELL 11: Clinical Dashboard — Pre vs. Post
# ============================================================
fig = plt.figure(figsize=(18, 14))
gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

one_pre  = pre[pre['time']  < GAIT_DURATION]
one_post = post[post['time'] < GAIT_DURATION]

jcols   = ['hip_angle', 'knee_angle', 'ankle_angle']
jlabels = ['Hip', 'Knee', 'Ankle']
jcolors = ['#e55', '#58a', '#2a7']

# Row 0: joint-by-joint overlay
for i, (col, label, col_color) in enumerate(zip(jcols, jlabels, jcolors)):
    ax = fig.add_subplot(gs[0, i])
    ax.plot(one_pre['gait_phase'],  one_pre[col],  'r--', lw=2, label='Pre-Tx',  alpha=0.9)
    ax.plot(one_post['gait_phase'], one_post[col], color=col_color, lw=2.5, label='Post-Tx')
    ax.fill_between(one_pre['gait_phase'], one_pre[col], one_post[col], alpha=0.15, color=col_color)
    ax.set_title(f'{label} Joint')
    ax.set_xlabel('Gait Cycle (%)')
    ax.set_ylabel('Angle (deg)')
    ax.legend(fontsize=8)

# Row 1: ROM comparison bar chart
ax_rom = fig.add_subplot(gs[1, :])
x = np.arange(len(jlabels))
w = 0.3
pre_roms  = [pre[c].max()  - pre[c].min()  for c in jcols]
post_roms = [post[c].max() - post[c].min() for c in jcols]
ax_rom.bar(x - w/2, pre_roms,  w, label='Pre-Tx',  color='salmon',    alpha=0.85, edgecolor='k', lw=0.5)
ax_rom.bar(x + w/2, post_roms, w, label='Post-Tx', color='steelblue', alpha=0.85, edgecolor='k', lw=0.5)
for xi, (pv, nv) in enumerate(zip(pre_roms, post_roms)):
    ax_rom.text(xi - w/2, pv + 0.3, f'{pv:.1f}', ha='center', va='bottom', fontsize=9)
    ax_rom.text(xi + w/2, nv + 0.3, f'{nv:.1f}', ha='center', va='bottom', fontsize=9)
ax_rom.set_xticks(x)
ax_rom.set_xticklabels(jlabels)
ax_rom.set_ylabel('ROM (degrees)')
ax_rom.set_title('Range of Motion: Pre vs. Post Treatment')
ax_rom.legend()

# Row 2 left: overall quality score
ax_q = fig.add_subplot(gs[2, 0])
bars = ax_q.bar(['Pre-Tx', 'Post-Tx'],
                [pre_scores['overall_score'], post_scores['overall_score']],
                color=['salmon', 'steelblue'], alpha=0.85, edgecolor='k', lw=0.5)
for bar, val in zip(bars, [pre_scores['overall_score'], post_scores['overall_score']]):
    ax_q.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
ax_q.set_ylim([0, 100])
ax_q.set_title('Overall Movement Quality Score')
ax_q.set_ylabel('Score (0–100)')

# Row 2 middle: ROM & Smoothness breakdown
ax_d = fig.add_subplot(gs[2, 1])
m_labels = ['ROM', 'Smoothness']
xm = np.arange(2)
ax_d.bar(xm - w/2, [pre_scores['rom_score'],  pre_scores['smoothness_score']],
         w, label='Pre-Tx',  color='salmon',    alpha=0.85)
ax_d.bar(xm + w/2, [post_scores['rom_score'], post_scores['smoothness_score']],
         w, label='Post-Tx', color='steelblue', alpha=0.85)
ax_d.set_xticks(xm)
ax_d.set_xticklabels(m_labels)
ax_d.set_ylim([0, 100])
ax_d.set_title('Quality Sub-Scores')
ax_d.legend(fontsize=8)

# Row 2 right: text summary
ax_s = fig.add_subplot(gs[2, 2])
ax_s.axis('off')
summary_lines = [
    'Treatment Effect Summary',
    '─' * 28,
    f'Overall improvement: {improvement:+.1f} pts',
    f'  ({improvement / pre_scores["overall_score"] * 100:+.1f}% change)',
    '',
    'ROM changes:',
    f'  Hip:   {post_roms[0]-pre_roms[0]:+.1f} deg',
    f'  Knee:  {post_roms[1]-pre_roms[1]:+.1f} deg',
    f'  Ankle: {post_roms[2]-pre_roms[2]:+.1f} deg',
]
ax_s.text(0.05, 0.95, '\n'.join(summary_lines),
          transform=ax_s.transAxes, va='top', fontsize=10,
          fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

fig.suptitle('Clinical Movement Assessment Dashboard — Pre vs. Post Treatment',
             fontsize=15, fontweight='bold', y=1.01)

save_path = f'{OUTPUT_DIR}/03_treatment_dashboard.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Dashboard saved: {save_path}')
plt.show()

pre.to_csv(f'{OUTPUT_DIR}/03_pre_treatment.csv', index=False)
post.to_csv(f'{OUTPUT_DIR}/03_post_treatment.csv', index=False)
quality_df.to_csv(f'{OUTPUT_DIR}/03_quality_scores.csv', index=False)
stat_df.to_csv(f'{OUTPUT_DIR}/03_statistical_comparison.csv', index=False)
print('All CSV files saved.')

In [ ]:
# ============================================================
# CELL 12: Final Summary
# ============================================================
print('=' * 52)
print('         WEEK 6 ANALYSIS COMPLETE')
print('=' * 52)
print(f'Output directory: {OUTPUT_DIR}')
print()
print('Generated files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f'  {f:<45} {size_kb:6.1f} KB')
print('=' * 52)

---
## Summary

### What this notebook covers

| Level | Topics |
|-------|--------|
| Beginner | Single joint angle (knee), ROM calculation, gait cycle visualisation |
| Intermediate | Multi-joint (hip, knee, ankle) kinematics, joint-specific ROM |
| Advanced | Pre/post treatment comparison, quality scoring, paired t-test, clinical dashboard |

### Clinical Applications
- **Diagnosis**: Detect restricted joint ROM
- **Monitoring**: Track rehabilitation progress over sessions
- **Outcome measurement**: Quantify treatment effectiveness
- **Documentation**: Export publication-ready figures and CSV reports

---
*Digital Healthcare & Physical Therapy — Week 6*